In [2]:
# ==============================================================================
# CELL 1: PHASE 1 - DATABASE SCHEMA & CONNECTION INITIALIZATION
# ==============================================================================
import sqlite3
import pandas as pd
from datetime import datetime

DB_NAME = "expense_tracker.db"

def get_connection():
    """Returns an active SQLite connection with Foreign Keys enforced."""
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_database():
    """Initializes the complete relational schema with strict constraints."""
    conn = get_connection()
    cursor = conn.cursor()
    
    # 1. Accounts Table (Sources of Funds)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS accounts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        type TEXT CHECK(type IN ('CASH', 'SAVINGS', 'CREDIT_CARD', 'CUSTOM')) NOT NULL,
        current_balance REAL NOT NULL DEFAULT 0.0,
        credit_limit REAL DEFAULT 0.0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """)

    # 2. Payment Methods Table (Routing Abstraction)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS payment_methods (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        default_account_id INTEGER,
        requires_account_prompt BOOLEAN DEFAULT 0,
        FOREIGN KEY (default_account_id) REFERENCES accounts(id) ON DELETE RESTRICT
    );
    """)

    # 3. Monthly Budgets Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS budgets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        category TEXT NOT NULL,
        monthly_limit REAL NOT NULL,
        month_year TEXT NOT NULL, -- Format: YYYY-MM
        UNIQUE(category, month_year)
    );
    """)

    # 4. Master Transactions Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type TEXT CHECK(type IN ('EXPENSE', 'INCOME', 'TRANSFER', 'SETTLEMENT')) NOT NULL,
        total_amount REAL NOT NULL,
        my_share_amount REAL NOT NULL,
        source_account_id INTEGER,
        destination_account_id INTEGER,
        payment_method_id INTEGER,
        category TEXT,
        notes TEXT,
        transaction_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (source_account_id) REFERENCES accounts(id) ON DELETE RESTRICT,
        FOREIGN KEY (destination_account_id) REFERENCES accounts(id) ON DELETE RESTRICT,
        FOREIGN KEY (payment_method_id) REFERENCES payment_methods(id) ON DELETE RESTRICT
    );
    """)

    # 5. Participants / Friends Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS friends (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        phone TEXT,
        upi_id TEXT
    );
    """)

    # 6. Expense Splits Table (Debt Ledger)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS expense_splits (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        transaction_id INTEGER NOT NULL,
        friend_id INTEGER NOT NULL,
        share_amount REAL NOT NULL,
        is_settled BOOLEAN DEFAULT 0,
        FOREIGN KEY (transaction_id) REFERENCES transactions(id) ON DELETE CASCADE,
        FOREIGN KEY (friend_id) REFERENCES friends(id) ON DELETE RESTRICT
    );
    """)

    # 7. Settlements Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS settlements (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        friend_id INTEGER NOT NULL,
        amount REAL NOT NULL,
        destination_account_id INTEGER NOT NULL,
        settled_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (friend_id) REFERENCES friends(id) ON DELETE RESTRICT,
        FOREIGN KEY (destination_account_id) REFERENCES accounts(id) ON DELETE RESTRICT
    );
    """)

    # 8. Mandates & Recurring Subscriptions Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS mandates (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        amount REAL NOT NULL,
        debit_day_of_month INTEGER NOT NULL CHECK(debit_day_of_month BETWEEN 1 AND 31),
        payment_method_id INTEGER NOT NULL,
        source_account_id INTEGER NOT NULL,
        category TEXT NOT NULL,
        is_active BOOLEAN DEFAULT 1,
        FOREIGN KEY (payment_method_id) REFERENCES payment_methods(id) ON DELETE RESTRICT,
        FOREIGN KEY (source_account_id) REFERENCES accounts(id) ON DELETE RESTRICT
    );
    """)

    # 9. Mandate Logs Table (Monthly Status Tracker)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS mandate_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        mandate_id INTEGER NOT NULL,
        month_year TEXT NOT NULL, -- Format: YYYY-MM
        status TEXT CHECK(status IN ('PENDING', 'DEBITED', 'SKIPPED')) DEFAULT 'PENDING',
        transaction_id INTEGER,
        debited_at TIMESTAMP,
        UNIQUE(mandate_id, month_year),
        FOREIGN KEY (mandate_id) REFERENCES mandates(id) ON DELETE CASCADE,
        FOREIGN KEY (transaction_id) REFERENCES transactions(id) ON DELETE SET NULL
    );
    """)

    conn.commit()
    conn.close()

# Run the initialization
init_database()

# Inspection Query: Print all created tables to verify
conn = get_connection()
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
conn.close()

print("=================================================================")
print("           PHASE 1: DATABASE INITIALIZATION REPORT               ")
print("=================================================================")
print(f"Database File: {DB_NAME}")
print(f"Total Relational Tables Created: {len(tables_df)}")
print("-----------------------------------------------------------------")
for idx, row in tables_df.iterrows():
    print(f" [{idx + 1}] Table: {row['name']}")
print("=================================================================")

           PHASE 1: DATABASE INITIALIZATION REPORT               
Database File: expense_tracker.db
Total Relational Tables Created: 9
-----------------------------------------------------------------
 [1] Table: accounts
 [2] Table: payment_methods
 [3] Table: budgets
 [4] Table: transactions
 [5] Table: friends
 [6] Table: expense_splits
 [7] Table: settlements
 [8] Table: mandates
 [9] Table: mandate_logs


In [4]:
# ==============================================================================
# CELL 2: PHASE 2 (REFINED) - YOUR REAL ACCOUNTS & PAYMENT ROUTING ENGINE
# ==============================================================================
import sqlite3
import pandas as pd
from tabulate import tabulate

def reset_and_seed_real_sources():
    """Resets accounts & payment methods and seeds your exact real-world financial structure."""
    conn = get_connection()
    cursor = conn.cursor()
    
    # Clean previous demo setup safely
    cursor.execute("DELETE FROM transactions;")
    cursor.execute("DELETE FROM payment_methods;")
    cursor.execute("DELETE FROM accounts;")
    
    # --------------------------------------------------------------------------
    # 1. YOUR REAL FINANCIAL SOURCES
    # --------------------------------------------------------------------------
    sources = [
        # (Name, Type, Initial Balance, Credit Limit)
        ("Cash Wallet", "CASH", 2000.0, 0.0),
        ("HDFC Savings (Daily)", "SAVINGS", 10000.0, 0.0),
        ("Indian Bank (EMI & Savings)", "SAVINGS", 15000.0, 0.0),
        ("PNB (Personal Savings)", "SAVINGS", 25000.0, 0.0),
        ("SBI Salary Account", "SAVINGS", 60000.0, 0.0),
        ("IDFC Credit Card", "CREDIT_CARD", 0.0, 35000.0),
        ("SBI Credit Card", "CREDIT_CARD", 0.0, 76000.0)
    ]
    
    account_map = {}
    for name, acc_type, bal, limit in sources:
        cursor.execute("""
            INSERT INTO accounts (name, type, current_balance, credit_limit)
            VALUES (?, ?, ?, ?)
        """, (name, acc_type, bal, limit))
        account_map[name] = cursor.lastrowid

    # --------------------------------------------------------------------------
    # 2. YOUR OPTIMIZED PAYMENT ROUTING METHODS
    # --------------------------------------------------------------------------
    methods = [
        # (Method Name, Default Account ID, Requires Prompt)
        ("Cash", account_map["Cash Wallet"], 0),
        ("GPay - HDFC", account_map["HDFC Savings (Daily)"], 0),
        ("GPay - SBI - CC", account_map["SBI Credit Card"], 0),
        ("GPay - IDFC - CC", account_map["IDFC Credit Card"], 0),
        ("NetBanking / AutoDebit - Indian Bank", account_map["Indian Bank (EMI & Savings)"], 0),
        ("NetBanking - SBI Salary", account_map["SBI Salary Account"], 0),
        ("Debit Card", None, 1)  # Requires choosing HDFC / Indian Bank / PNB / SBI
    ]
    
    for method_name, default_acc, prompt in methods:
        cursor.execute("""
            INSERT INTO payment_methods (name, default_account_id, requires_account_prompt)
            VALUES (?, ?, ?)
        """, (method_name, default_acc, prompt))

    conn.commit()
    conn.close()

def display_accounts_and_methods():
    """Renders formatted tables of active accounts, live balances, and payment mappings."""
    conn = get_connection()
    
    accounts_query = """
        SELECT 
            id, 
            name AS 'Account Name', 
            type AS 'Type', 
            printf('₹%.2f', current_balance) AS 'Current Balance', 
            printf('₹%.2f', credit_limit) AS 'Credit Limit'
        FROM accounts
    """
    accounts_df = pd.read_sql_query(accounts_query, conn)
    
    methods_query = """
        SELECT 
            pm.id, 
            pm.name AS 'Payment Method', 
            COALESCE(a.name, 'Prompt on checkout') AS 'Linked Funding Source'
        FROM payment_methods pm
        LEFT JOIN accounts a ON pm.default_account_id = a.id
    """
    methods_df = pd.read_sql_query(methods_query, conn)
    conn.close()
    
    print("=================================================================")
    print("        PHASE 2: REAL ACCOUNTS & PAYMENT ROUTING ENGINE          ")
    print("=================================================================")
    print("\n--- YOUR CONFIGURED FINANCIAL SOURCES ---")
    print(tabulate(accounts_df, headers='keys', tablefmt='psql', showindex=False))
    print("\n--- OPTIMIZED PAYMENT METHODS & ROUTING ---")
    print(tabulate(methods_df, headers='keys', tablefmt='psql', showindex=False))
    print("=================================================================")

# Execute reset and display
reset_and_seed_real_sources()
display_accounts_and_methods()

        PHASE 2: REAL ACCOUNTS & PAYMENT ROUTING ENGINE          

--- YOUR CONFIGURED FINANCIAL SOURCES ---
+------+-----------------------------+-------------+-------------------+----------------+
|   id | Account Name                | Type        | Current Balance   | Credit Limit   |
|------+-----------------------------+-------------+-------------------+----------------|
|    7 | Cash Wallet                 | CASH        | ₹2000.00          | ₹0.00          |
|    8 | HDFC Savings (Daily)        | SAVINGS     | ₹10000.00         | ₹0.00          |
|    9 | Indian Bank (EMI & Savings) | SAVINGS     | ₹15000.00         | ₹0.00          |
|   10 | PNB (Personal Savings)      | SAVINGS     | ₹25000.00         | ₹0.00          |
|   11 | SBI Salary Account          | SAVINGS     | ₹60000.00         | ₹0.00          |
|   12 | IDFC Credit Card            | CREDIT_CARD | ₹0.00             | ₹35000.00      |
|   13 | SBI Credit Card             | CREDIT_CARD | ₹0.00             | ₹76000.0

In [5]:
# ==============================================================================
# CELL 3: PHASE 3 - REAL-TIME EXPENSE LOGGING & BUDGET DEDUCTION ENGINE
# ==============================================================================
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

def set_monthly_budget(category: str, limit_amount: float, month_year: str = None):
    """Sets or updates a monthly budget cap for a specific category."""
    if month_year is None:
        month_year = datetime.now().strftime("%Y-%m")
    
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO budgets (category, monthly_limit, month_year)
        VALUES (?, ?, ?)
        ON CONFLICT(category, month_year) DO UPDATE SET monthly_limit = excluded.monthly_limit
    """, (category, limit_amount, month_year))
    conn.commit()
    conn.close()

def log_expense(amount: float, category: str, payment_method_id: int, 
                override_account_id: int = None, notes: str = "", transaction_date: str = None):
    """
    Logs an expense atomically:
    1. Resolves funding account from payment method.
    2. Debits the source balance (or increases credit card dues).
    3. Writes the transaction record.
    4. Evaluates real-time category budget consumption and warns if near/over limit.
    """
    if amount <= 0:
        raise ValueError("Expense amount must be greater than 0.")
    
    if transaction_date is None:
        transaction_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    month_year = transaction_date[:7]
    conn = get_connection()
    cursor = conn.cursor()

    try:
        # 1. Resolve funding account
        cursor.execute("SELECT default_account_id, requires_account_prompt, name FROM payment_methods WHERE id = ?", (payment_method_id,))
        method = cursor.fetchone()
        if not method:
            raise ValueError(f"Payment Method ID {payment_method_id} does not exist.")
        
        default_acc_id, requires_prompt, method_name = method
        
        if requires_prompt:
            if not override_account_id:
                raise ValueError(f"Payment method '{method_name}' requires an explicit source account selection.")
            source_acc_id = override_account_id
        else:
            source_acc_id = default_acc_id

        # 2. Atomic account balance deduction
        cursor.execute("UPDATE accounts SET current_balance = current_balance - ? WHERE id = ?", (amount, source_acc_id))

        # 3. Master transaction record
        cursor.execute("""
            INSERT INTO transactions (type, total_amount, my_share_amount, source_account_id, payment_method_id, category, notes, transaction_date)
            VALUES ('EXPENSE', ?, ?, ?, ?, ?, ?, ?)
        """, (amount, amount, source_acc_id, payment_method_id, category, notes, transaction_date))
        tx_id = cursor.lastrowid

        # 4. Check budget threshold
        cursor.execute("SELECT monthly_limit FROM budgets WHERE category = ? AND month_year = ?", (category, month_year))
        budget_row = cursor.fetchone()
        
        alert_msg = None
        if budget_row:
            monthly_limit = budget_row[0]
            cursor.execute("""
                SELECT COALESCE(SUM(my_share_amount), 0.0) 
                FROM transactions 
                WHERE type = 'EXPENSE' AND category = ? AND strftime('%Y-%m', transaction_date) = ?
            """, (category, month_year))
            total_spent = cursor.fetchone()[0]
            pct = (total_spent / monthly_limit) * 100

            if pct >= 100:
                alert_msg = f"⚠️ BUDGET BREACH: {category} spend is at {pct:.1f}% (₹{total_spent:,.2f} / ₹{monthly_limit:,.2f})"
            elif pct >= 80:
                alert_msg = f"⚡ BUDGET WARNING: {category} spend has reached {pct:.1f}% (₹{total_spent:,.2f} / ₹{monthly_limit:,.2f})"

        conn.commit()
        return tx_id, alert_msg
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

def display_budget_and_expenses(month_year: str = None):
    """Renders formatted tables showing real-time budget status and recent transactions."""
    if month_year is None:
        month_year = datetime.now().strftime("%Y-%m")
    
    conn = get_connection()
    
    budget_query = f"""
        SELECT 
            b.category AS 'Category',
            printf('₹%.2f', b.monthly_limit) AS 'Allocated Limit',
            printf('₹%.2f', COALESCE(spent.total_spent, 0.0)) AS 'Spent So Far',
            printf('₹%.2f', b.monthly_limit - COALESCE(spent.total_spent, 0.0)) AS 'Remaining Balance',
            printf('%.1f%%', (COALESCE(spent.total_spent, 0.0) / b.monthly_limit) * 100) AS 'Consumed %'
        FROM budgets b
        LEFT JOIN (
            SELECT category, SUM(my_share_amount) AS total_spent
            FROM transactions
            WHERE type = 'EXPENSE' AND strftime('%Y-%m', transaction_date) = '{month_year}'
            GROUP BY category
        ) spent ON b.category = spent.category
        WHERE b.month_year = '{month_year}'
    """
    budget_df = pd.read_sql_query(budget_query, conn)
    
    tx_query = """
        SELECT 
            t.id, 
            t.transaction_date AS 'Date',
            t.category AS 'Category',
            printf('₹%.2f', t.total_amount) AS 'Amount',
            a.name AS 'Debited Source',
            pm.name AS 'Payment Method',
            t.notes AS 'Notes'
        FROM transactions t
        LEFT JOIN accounts a ON t.source_account_id = a.id
        LEFT JOIN payment_methods pm ON t.payment_method_id = pm.id
        WHERE t.type = 'EXPENSE'
        ORDER BY t.id DESC
        LIMIT 5
    """
    tx_df = pd.read_sql_query(tx_query, conn)
    conn.close()

    print("=================================================================")
    print(f"       PHASE 3: EXPENSE & BUDGET DASHBOARD ({month_year})        ")
    print("=================================================================")
    print("\n--- BUDGET CONSUMPTION RUNWAY ---")
    print(tabulate(budget_df, headers='keys', tablefmt='psql', showindex=False))
    print("\n--- RECENT PERSONAL TRANSACTIONS ---")
    print(tabulate(tx_df, headers='keys', tablefmt='psql', showindex=False))
    print("=================================================================")

# ------------------------------------------------------------------------------
# SIMULATING BUDGET SETUP & REAL TRANSACTIONS
# ------------------------------------------------------------------------------
current_month = datetime.now().strftime("%Y-%m")

# Set Category Budgets
set_monthly_budget("Food & Dining", 8000.0, current_month)
set_monthly_budget("Groceries", 6000.0, current_month)
set_monthly_budget("Fuel & Travel", 4000.0, current_month)

# 1. Groceries via GPay - SBI - CC (Method ID: 9 -> Dues increase on SBI CC)
tx1, alert1 = log_expense(
    amount=2400.0, 
    category="Groceries", 
    payment_method_id=9, 
    notes="Monthly supermarket run via RuPay CC"
)
if alert1: print(alert1)

# 2. Daily Tea/Snack via GPay - HDFC (Method ID: 8 -> Debits HDFC Savings)
tx2, alert2 = log_expense(
    amount=180.0, 
    category="Food & Dining", 
    payment_method_id=8, 
    notes="Evening snacks"
)
if alert2: print(alert2)

# 3. Fuel via Debit Card (Method ID: 13 -> Prompts Bank -> Selects HDFC Savings Account ID: 8)
tx3, alert3 = log_expense(
    amount=3300.0, 
    category="Fuel & Travel", 
    payment_method_id=13, 
    override_account_id=8,
    notes="Petrol pump fuel fill"
)
if alert3: print(alert3)

# Display state
display_budget_and_expenses(current_month)

⚡ BUDGET WARNING: Fuel & Travel spend has reached 82.5% (₹3,300.00 / ₹4,000.00)
       PHASE 3: EXPENSE & BUDGET DASHBOARD (2026-10)        

--- BUDGET CONSUMPTION RUNWAY ---
+---------------+-------------------+----------------+---------------------+--------------+
| Category      | Allocated Limit   | Spent So Far   | Remaining Balance   | Consumed %   |
|---------------+-------------------+----------------+---------------------+--------------|
| Food & Dining | ₹8000.00          | ₹180.00        | ₹7820.00            | 2.3%         |
| Groceries     | ₹6000.00          | ₹2400.00       | ₹3600.00            | 40.0%        |
| Fuel & Travel | ₹4000.00          | ₹3300.00       | ₹700.00             | 82.5%        |
+---------------+-------------------+----------------+---------------------+--------------+

--- RECENT PERSONAL TRANSACTIONS ---
+------+---------------------+---------------+----------+----------------------+------------------+--------------------------------------+
|  

In [6]:
# ==============================================================================
# CELL 4: PHASE 4 - BILL SPLITTING & "WHO OWES WHOM" DEBT ENGINE
# ==============================================================================
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

def add_friend(name: str, phone: str = None, upi_id: str = None):
    """Registers a friend to participate in bill splitting."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO friends (name, phone, upi_id)
        VALUES (?, ?, ?)
    """, (name, phone, upi_id))
    conn.commit()
    friend_id = cursor.lastrowid
    conn.close()
    return friend_id

def log_split_expense(total_amount: float, category: str, payment_method_id: int, 
                      my_share: float, friend_splits: list, override_account_id: int = None, 
                      notes: str = ""):
    """
    Logs a multi-person split bill atomically:
    1. Entire bill amount is debited from your selected funding source.
    2. Only 'my_share' impacts your personal category budget.
    3. Remaining shares are added as debt entries against respective friends.
    friend_splits format: [(friend_id_1, share_amount_1), (friend_id_2, share_amount_2)]
    """
    # Validation: Ensure individual shares sum up exactly to the total bill
    total_friends_share = sum(share for _, share in friend_splits)
    if round(my_share + total_friends_share, 2) != round(total_amount, 2):
        raise ValueError(f"Sum of shares (₹{my_share + total_friends_share}) must equal total amount (₹{total_amount}).")

    conn = get_connection()
    cursor = conn.cursor()
    try:
        # Resolve source account
        cursor.execute("SELECT default_account_id, requires_account_prompt, name FROM payment_methods WHERE id = ?", (payment_method_id,))
        method = cursor.fetchone()
        default_acc_id, requires_prompt, method_name = method
        
        source_acc_id = override_account_id if requires_prompt else default_acc_id

        # 1. Deduct full total_amount from your account
        cursor.execute("UPDATE accounts SET current_balance = current_balance - ? WHERE id = ?", (total_amount, source_acc_id))

        # 2. Insert master transaction with my_share_amount (budget only tracks your share)
        cursor.execute("""
            INSERT INTO transactions (type, total_amount, my_share_amount, source_account_id, payment_method_id, category, notes)
            VALUES ('EXPENSE', ?, ?, ?, ?, ?, ?)
        """, (total_amount, my_share, source_acc_id, payment_method_id, category, notes))
        tx_id = cursor.lastrowid

        # 3. Create debt entries in expense_splits
        for friend_id, share in friend_splits:
            cursor.execute("""
                INSERT INTO expense_splits (transaction_id, friend_id, share_amount, is_settled)
                VALUES (?, ?, ?, 0)
            """, (tx_id, friend_id, share))

        conn.commit()
        return tx_id
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

def settle_up_with_friend(friend_id: int, amount: float, destination_account_id: int):
    """
    Records repayment from a friend:
    1. Marks pending debt as settled.
    2. Directly credits the received money to your chosen destination account.
    """
    conn = get_connection()
    cursor = conn.cursor()
    try:
        # 1. Credit destination account (e.g. HDFC Savings)
        cursor.execute("UPDATE accounts SET current_balance = current_balance + ? WHERE id = ?", (amount, destination_account_id))

        # 2. Record settlement
        cursor.execute("""
            INSERT INTO settlements (friend_id, amount, destination_account_id)
            VALUES (?, ?, ?)
        """, (friend_id, amount, destination_account_id))

        # 3. Mark split rows as settled for this friend
        cursor.execute("""
            UPDATE expense_splits 
            SET is_settled = 1 
            WHERE friend_id = ? AND is_settled = 0
        """, (friend_id,))

        conn.commit()
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

def display_debt_matrix():
    """Renders the 'Who Owes Whom' ledger showing net balances for every friend."""
    conn = get_connection()
    
    query = """
        SELECT 
            f.id,
            f.name AS 'Friend Name',
            COALESCE(owed.total_owed, 0.0) AS 'Total Split Share',
            COALESCE(settled.total_settled, 0.0) AS 'Total Settled Up',
            (COALESCE(owed.total_owed, 0.0) - COALESCE(settled.total_settled, 0.0)) AS 'Net Balance'
        FROM friends f
        LEFT JOIN (
            SELECT friend_id, SUM(share_amount) AS total_owed
            FROM expense_splits
            GROUP BY friend_id
        ) owed ON f.id = owed.friend_id
        LEFT JOIN (
            SELECT friend_id, SUM(amount) AS total_settled
            FROM settlements
            GROUP BY friend_id
        ) settled ON f.id = settled.friend_id
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    # Format visual indicators
    df['Status'] = df['Net Balance'].apply(
        lambda bal: f"🟢 Owes you ₹{bal:,.2f}" if bal > 0 else ("🔴 You owe" if bal < 0 else "⚪ All Settled")
    )
    df['Total Split Share'] = df['Total Split Share'].apply(lambda x: f"₹{x:,.2f}")
    df['Total Settled Up'] = df['Total Settled Up'].apply(lambda x: f"₹{x:,.2f}")
    df['Net Balance'] = df['Net Balance'].apply(lambda x: f"₹{x:,.2f}")

    print("=================================================================")
    print("      PHASE 4: SPLIT-BILL & 'WHO OWES WHOM' DEBT MATRIX          ")
    print("=================================================================")
    print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))
    print("=================================================================")

# ------------------------------------------------------------------------------
# SIMULATING REAL-WORLD BILL SPLIT & SETTLE-UP SCENARIOS
# ------------------------------------------------------------------------------
# 1. Add your friends
rohan_id = add_friend("Rohan", phone="9876543210", upi_id="rohan@okaxis")
aman_id = add_friend("Aman", phone="9812345678", upi_id="aman@okhdfcbank")

# 2. Bill Split Scenario 1:
# You pay ₹3,000 for dinner for 3 people (You, Rohan, Aman) using GPay - SBI - CC (Method ID: 9)
# Equal split: ₹1,000 each.
log_split_expense(
    total_amount=3000.0,
    category="Food & Dining",
    payment_method_id=9, # GPay - SBI - CC
    my_share=1000.0,
    friend_splits=[(rohan_id, 1000.0), (aman_id, 1000.0)],
    notes="Weekend team dinner"
)

# 3. Bill Split Scenario 2:
# You pay ₹600 for movie snacks using GPay - HDFC (Method ID: 8)
# Split: Rohan owes ₹300, You take ₹300.
log_split_expense(
    total_amount=600.0,
    category="Food & Dining",
    payment_method_id=8, # GPay - HDFC
    my_share=300.0,
    friend_splits=[(rohan_id, 300.0)],
    notes="Movie popcorn"
)

# 4. Settle-Up Scenario:
# Rohan transfers ₹1,000 back to you on Google Pay (lands in your HDFC Savings Account ID: 8)
settle_up_with_friend(
    friend_id=rohan_id, 
    amount=1000.0, 
    destination_account_id=8 # HDFC Savings Account
)

# Display the debt matrix
display_debt_matrix()

      PHASE 4: SPLIT-BILL & 'WHO OWES WHOM' DEBT MATRIX          
+------+---------------+---------------------+--------------------+---------------+-----------------------+
|   id | Friend Name   | Total Split Share   | Total Settled Up   | Net Balance   | Status                |
|------+---------------+---------------------+--------------------+---------------+-----------------------|
|    2 | Aman          | ₹1,000.00           | ₹0.00              | ₹1,000.00     | 🟢 Owes you ₹1,000.00 |
|    1 | Rohan         | ₹1,300.00           | ₹1,000.00          | ₹300.00       | 🟢 Owes you ₹300.00   |
+------+---------------+---------------------+--------------------+---------------+-----------------------+


In [7]:
# ==============================================================================
# CELL 5: PHASE 5 - MANDATES & RECURRING SUBSCRIPTIONS MANAGER
# ==============================================================================
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

def create_mandate(name: str, amount: float, debit_day: int, payment_method_id: int, 
                   category: str, source_account_id: int = None):
    """Registers a recurring auto-debit or mandate."""
    conn = get_connection()
    cursor = conn.cursor()
    
    # Resolve default source account if not passed
    if source_account_id is None:
        cursor.execute("SELECT default_account_id FROM payment_methods WHERE id = ?", (payment_method_id,))
        source_account_id = cursor.fetchone()[0]
        
    cursor.execute("""
        INSERT INTO mandates (name, amount, debit_day_of_month, payment_method_id, source_account_id, category, is_active)
        VALUES (?, ?, ?, ?, ?, ?, 1)
    """, (name, amount, debit_day, payment_method_id, source_account_id, category))
    mandate_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return mandate_id

def edit_mandate(mandate_id: int, name: str = None, amount: float = None, 
                 debit_day: int = None, payment_method_id: int = None, 
                 source_account_id: int = None):
    """
    Edits an existing mandate's details: name, amount, debit date, or source account.
    """
    conn = get_connection()
    cursor = conn.cursor()
    
    updates = []
    params = []
    if name is not None:
        updates.append("name = ?")
        params.append(name)
    if amount is not None:
        updates.append("amount = ?")
        params.append(amount)
    if debit_day is not None:
        updates.append("debit_day_of_month = ?")
        params.append(debit_day)
    if payment_method_id is not None:
        updates.append("payment_method_id = ?")
        params.append(payment_method_id)
    if source_account_id is not None:
        updates.append("source_account_id = ?")
        params.append(source_account_id)
        
    params.append(mandate_id)
    cursor.execute(f"UPDATE mandates SET {', '.join(updates)} WHERE id = ?", params)
    conn.commit()
    conn.close()

def sync_monthly_mandate_logs(month_year: str = None):
    """Generates monthly pending checklist entries for all active mandates."""
    if month_year is None:
        month_year = datetime.now().strftime("%Y-%m")
        
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM mandates WHERE is_active = 1")
    active_mandates = cursor.fetchall()
    
    for (m_id,) in active_mandates:
        cursor.execute("""
            INSERT OR IGNORE INTO mandate_logs (mandate_id, month_year, status)
            VALUES (?, ?, 'PENDING')
        """, (m_id, month_year))
        
    conn.commit()
    conn.close()

def mark_mandate_debited(mandate_id: int, month_year: str = None, actual_debit_date: str = None):
    """
    Executes a mandate debit:
    1. Flips monthly status from PENDING to DEBITED.
    2. Writes an EXPENSE transaction.
    3. Deducts from the configured source account and updates category budget.
    """
    if month_year is None:
        month_year = datetime.now().strftime("%Y-%m")
    if actual_debit_date is None:
        actual_debit_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
    conn = get_connection()
    cursor = conn.cursor()
    try:
        # Fetch mandate details
        cursor.execute("""
            SELECT name, amount, payment_method_id, source_account_id, category 
            FROM mandates WHERE id = ?
        """, (mandate_id,))
        mandate = cursor.fetchone()
        name, amount, method_id, source_acc_id, category = mandate

        # 1. Deduct from source account
        cursor.execute("UPDATE accounts SET current_balance = current_balance - ? WHERE id = ?", (amount, source_acc_id))

        # 2. Log transaction
        cursor.execute("""
            INSERT INTO transactions (type, total_amount, my_share_amount, source_account_id, payment_method_id, category, notes, transaction_date)
            VALUES ('EXPENSE', ?, ?, ?, ?, ?, ?, ?)
        """, (amount, amount, source_acc_id, method_id, category, f"Auto-debit mandate: {name}", actual_debit_date))
        tx_id = cursor.lastrowid

        # 3. Update monthly mandate log
        cursor.execute("""
            UPDATE mandate_logs 
            SET status = 'DEBITED', transaction_id = ?, debited_at = ?
            WHERE mandate_id = ? AND month_year = ?
        """, (tx_id, actual_debit_date, mandate_id, month_year))

        conn.commit()
        return tx_id
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

def display_mandates_dashboard(month_year: str = None):
    """Renders the active mandates configuration and current monthly status."""
    if month_year is None:
        month_year = datetime.now().strftime("%Y-%m")
        
    conn = get_connection()
    
    query = f"""
        SELECT 
            m.id,
            m.name AS 'Mandate / Subscription',
            printf('₹%.2f', m.amount) AS 'Amount',
            printf('%02dth of month', m.debit_day_of_month) AS 'Debit Day',
            a.name AS 'Debited Source Account',
            pm.name AS 'Payment Route',
            COALESCE(ml.status, 'NOT GENERATED') AS 'Monthly Status'
        FROM mandates m
        LEFT JOIN accounts a ON m.source_account_id = a.id
        LEFT JOIN payment_methods pm ON m.payment_method_id = pm.id
        LEFT JOIN mandate_logs ml ON m.id = ml.mandate_id AND ml.month_year = '{month_year}'
        WHERE m.is_active = 1
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    df['Monthly Status'] = df['Monthly Status'].apply(
        lambda s: "🟢 DEBITED" if s == 'DEBITED' else ("🟡 PENDING" if s == 'PENDING' else s)
    )

    print("=================================================================")
    print(f"      PHASE 5: RECURRING MANDATES DASHBOARD ({month_year})       ")
    print("=================================================================")
    print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))
    print("=================================================================")

# ------------------------------------------------------------------------------
# SIMULATING MANDATES CREATION, EDITING & DEBIT EXECUTION
# ------------------------------------------------------------------------------
current_month = datetime.now().strftime("%Y-%m")

# 1. Register Mandates
# A. Education Loan EMI on Indian Bank (Account ID: 9, Method ID: 11)
emi_id = create_mandate(
    name="Education Loan EMI",
    amount=7500.0,
    debit_day=10,
    payment_method_id=11, # NetBanking / AutoDebit - Indian Bank
    category="Loans & EMIs",
    source_account_id=9   # Indian Bank (EMI & Savings)
)

# B. Netflix Subscription on SBI Credit Card (Account ID: 13, Method ID: 9)
netflix_id = create_mandate(
    name="Netflix Subscription",
    amount=649.0,
    debit_day=5,
    payment_method_id=9,  # GPay - SBI - CC
    category="Subscriptions",
    source_account_id=13  # SBI Credit Card
)

# 2. Simulate User Editing Mandate:
# Suppose Netflix price changes from ₹649 to ₹499 and user moves it to IDFC CC (Account ID: 12, Method ID: 10)
edit_mandate(
    mandate_id=netflix_id,
    name="Netflix Standard Plan",
    amount=499.0,
    payment_method_id=10, # GPay - IDFC - CC
    source_account_id=12  # IDFC Credit Card
)

# 3. Initialize current month tracking
sync_monthly_mandate_logs(current_month)

# 4. Simulate Auto-Debit execution:
# Education Loan EMI gets debited today
mark_mandate_debited(mandate_id=emi_id, month_year=current_month)

# 5. Display the Mandates dashboard
display_mandates_dashboard(current_month)

      PHASE 5: RECURRING MANDATES DASHBOARD (2026-10)       
+------+--------------------------+----------+---------------+-----------------------------+--------------------------------------+------------------+
|   id | Mandate / Subscription   | Amount   | Debit Day     | Debited Source Account      | Payment Route                        | Monthly Status   |
|------+--------------------------+----------+---------------+-----------------------------+--------------------------------------+------------------|
|    1 | Education Loan EMI       | ₹7500.00 | 10th of month | Indian Bank (EMI & Savings) | NetBanking / AutoDebit - Indian Bank | 🟢 DEBITED       |
|    2 | Netflix Standard Plan    | ₹499.00  | 05th of month | IDFC Credit Card            | GPay - IDFC - CC                     | 🟡 PENDING       |
+------+--------------------------+----------+---------------+-----------------------------+--------------------------------------+------------------+
